In [ ]:
"""Minimum-time drone racing with GPU-SLS in an RTI-MPC loop."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys
from time import perf_counter
import copy
import jax
import jax.numpy as jnp
import numpy as np

from crazyflow.dynamics.core import load_params, parametrize
from crazyflow.dynamics.first_principles import dynamics as crazyflow_dynamics
from drone_sim import DRONE_MODEL, GATES, close, dynamics, get_sim, render, step
from gpu_sls.generic_mpc import GenericMPC, MPCConfig
from gpu_sls.gpu_admm import ADMMConfig
from gpu_sls.gpu_sls import SLSConfig
from gpu_sls.gpu_sqp import SQPConfig
from gpu_sls.utils.constraint_utils import (
    combine_constraints,
    make_control_box_constraints,
    make_state_box_constraints,
)


# Planner state: [pos(3), quat_xyzw(4), vel(3), body_rates(3),
#                 rotor_rpm(4), total_horizon_time]
NX = 18
NU = 4
HORIZON = 30
MAX_RTI_STEPS = 250
GOAL_TOLERANCE = 0.20
GATE_TOLERANCE = 0.55
MIN_DURATION = 0.4
MAX_DURATION = 8.0
MAX_ROTOR_RPM = 25_000.0
MODEL_SUBSTEPS = 5

FIRST_PRINCIPLES = parametrize(crazyflow_dynamics, DRONE_MODEL, xp=jnp)
FIRST_PRINCIPLES_PARAMS = load_params(crazyflow_dynamics, DRONE_MODEL)


def _hover_rpm() -> float:
    coefficients = np.asarray(FIRST_PRINCIPLES_PARAMS["rpm2thrust"], dtype=float).copy()
    coefficients[0] -= (
        float(FIRST_PRINCIPLES_PARAMS["mass"]) * 9.81 / 4.0
    )
    roots = np.roots(coefficients[::-1])
    positive_roots = roots[np.isreal(roots) & (roots.real > 0)].real
    return float(positive_roots.min())


HOVER_RPM = _hover_rpm()
HOVER_COMMAND = HOVER_RPM / MAX_ROTOR_RPM


def initialize_hover(position=(0.0, 0.0, 0.75)):
    """Reset Crazyflow to a stationary airborne hover equilibrium."""
    sim = get_sim()
    sim.reset()
    states = sim.data.states
    hover_position = jnp.asarray(position, dtype=states.pos.dtype)
    identity_quaternion = jnp.asarray(
        [0.0, 0.0, 0.0, 1.0], dtype=states.quat.dtype
    )
    states = states.replace(
        pos=states.pos.at[...].set(hover_position),
        quat=states.quat.at[...].set(identity_quaternion),
        vel=jnp.zeros_like(states.vel),
        ang_vel=jnp.zeros_like(states.ang_vel),
        rotor_vel=jnp.full_like(states.rotor_vel, HOVER_RPM),
    )
    sim.data = sim.data.replace(
        states=states,
        core=sim.data.core.replace(
            mjx_synced=jnp.zeros_like(sim.data.core.mjx_synced)
        ),
    )
    dynamics(np.full(NU, HOVER_RPM, dtype=np.float32))
    sim.default_data = sim.data.replace()
    return sim


@dataclass(frozen=True)
class RTIResult:
    reached_goal: bool
    iterations: int
    simulated_time: float
    final_position: np.ndarray


def min_time_dynamics(
    x: jax.Array,
    u: jax.Array,
    _t: jax.Array,
    *,
    parameter: float,
) -> jax.Array:
    """Crazyflow's first-principles model with rotor and quaternion dynamics."""
    dt = parameter * x[-1] / MODEL_SUBSTEPS

    def integrate_substep(_index: int, state: jax.Array) -> jax.Array:
        pos = state[:3]
        quat = state[3:7]
        vel = state[7:10]
        ang_vel = state[10:13]
        rotor_command = u * MAX_ROTOR_RPM
        rotor_vel = state[13:17] * MAX_ROTOR_RPM
        pos_dot, _, vel_dot, ang_vel_dot, rotor_vel_dot = FIRST_PRINCIPLES(
            pos=pos,
            quat=quat,
            vel=vel,
            ang_vel=ang_vel,
            cmd=rotor_command,
            rotor_vel=rotor_vel,
        )

        rotation_vector = ang_vel * dt
        # The epsilon keeps the Jacobian finite at zero body rate.
        angle = jnp.sqrt(jnp.dot(rotation_vector, rotation_vector) + 1e-12)
        delta_quat = jnp.concatenate(
            (
                0.5
                * jnp.sinc(angle / (2.0 * jnp.pi))
                * rotation_vector,
                jnp.cos(0.5 * angle)[None],
            )
        )
        q_xyz, q_w = quat[:3], quat[3]
        d_xyz, d_w = delta_quat[:3], delta_quat[3]
        next_quat = jnp.concatenate(
            (
                q_w * d_xyz + d_w * q_xyz + jnp.cross(q_xyz, d_xyz),
                (q_w * d_w - jnp.dot(q_xyz, d_xyz))[None],
            )
        )
        next_quat = next_quat / jnp.sqrt(jnp.dot(next_quat, next_quat) + 1e-12)
        return jnp.concatenate(
            (
                pos + dt * pos_dot,
                next_quat,
                vel + dt * vel_dot,
                ang_vel + dt * ang_vel_dot,
                state[13:17] + dt * rotor_vel_dot / MAX_ROTOR_RPM,
                state[-1:],
            )
        )

    return jax.lax.fori_loop(0, MODEL_SUBSTEPS, integrate_substep, x)


def min_time_cost(
    weights: jax.Array,
    reference: jax.Array,
    x: jax.Array,
    u: jax.Array,
    t: jax.Array,
) -> jax.Array:
    position_error = x[:3] - reference[t, :3]
    quaternion_alignment = jnp.dot(x[3:7], reference[t, 3:7])
    velocity_error = x[7:10] - reference[t, 7:10]
    body_rate_error = x[10:13] - reference[t, 10:13]
    rotor_command_error = u - HOVER_COMMAND
    return (
        weights[0] * jnp.dot(position_error, position_error)
        + weights[1] * (1.0 - quaternion_alignment**2)
        + weights[2] * jnp.dot(velocity_error, velocity_error)
        + weights[3] * jnp.dot(body_rate_error, body_rate_error)
        + weights[4] * jnp.dot(rotor_command_error, rotor_command_error)
        + weights[5] * x[-1]
    )


def make_terminal_constraint(goal: jax.Array, half_width: float):
    goal = jnp.asarray(goal)

    def constraint(x: jax.Array, _u: jax.Array, t: jax.Array) -> jax.Array:
        error = x[:3] - goal
        terminal = jnp.concatenate((error - half_width, -error - half_width))
        return jnp.where(t == HORIZON, terminal, -jnp.ones_like(terminal))

    return constraint


def disturbance(x_trajectory: jax.Array) -> jax.Array:
    """Small process-noise set used for SLS tube construction."""
    dt = x_trajectory[:, -1] / HORIZON
    identity = jnp.eye(NX, dtype=x_trajectory.dtype)
    matrices = 1e-3 * dt[:, None, None] * identity[None, :, :]
    return matrices.at[:, -1, -1].set(0.0)


def build_reference(
    x0: jax.Array,
    waypoints: np.ndarray,
    duration: float,
) -> jax.Array:
    """Sample a constant-speed polyline through every remaining gate."""
    points = np.vstack((np.asarray(x0[:3]), np.asarray(waypoints)))
    segment_vectors = np.diff(points, axis=0)
    segment_lengths = np.linalg.norm(segment_vectors, axis=1)
    cumulative = np.concatenate(([0.0], np.cumsum(segment_lengths)))
    total_length = max(float(cumulative[-1]), 1e-6)
    samples = np.linspace(0.0, total_length, HORIZON + 1)
    positions = np.empty((HORIZON + 1, 3), dtype=np.float32)

    for index, distance in enumerate(samples):
        segment = min(np.searchsorted(cumulative, distance, side="right") - 1, len(segment_lengths) - 1)
        segment = max(segment, 0)
        length = max(float(segment_lengths[segment]), 1e-6)
        alpha = (distance - cumulative[segment]) / length
        positions[index] = points[segment] + alpha * segment_vectors[segment]

    dt = max(duration / HORIZON, 1e-3)
    velocities = np.gradient(positions, dt, axis=0)
    reference = np.zeros((HORIZON + 1, NX), dtype=np.float32)
    reference[:, :3] = positions
    reference[:, 6] = 1.0  # identity quaternion in xyzw convention
    reference[:, 7:10] = velocities
    reference[:, 13:17] = HOVER_COMMAND
    reference[:, -1] = duration
    return jnp.asarray(reference)


def measured_state(duration: float) -> jax.Array:
    sim = get_sim()
    position = np.asarray(sim.data.states.pos[0, 0])
    quaternion = np.asarray(sim.data.states.quat[0, 0])
    velocity = np.asarray(sim.data.states.vel[0, 0])
    angular_velocity = np.asarray(sim.data.states.ang_vel[0, 0])
    rotor_velocity = np.asarray(sim.data.states.rotor_vel[0, 0])
    return jnp.asarray(
        np.concatenate(
            (
                position,
                quaternion,
                velocity,
                angular_velocity,
                rotor_velocity / MAX_ROTOR_RPM,
                [duration],
            )
        ),
        dtype=jnp.float32,
    )


def build_controller(x0: jax.Array, reference: jax.Array, goal: jax.Array) -> GenericMPC:
    weights = jnp.array([0.01, 0.01, 0.01, 0.001, 0.001, 1.0], dtype=jnp.float32)
    config = MPCConfig(
        n=NX,
        nu=NU,
        N=HORIZON,
        W=weights,
        u_ref=jnp.full((NU,), HOVER_COMMAND, dtype=jnp.float32),
    )

    u_min = jnp.zeros(NU, dtype=jnp.float32)
    u_max = jnp.ones(NU, dtype=jnp.float32)
    x_min = jnp.array(
        [
            -20.0, -20.0, 0.0,
            -1.0, -1.0, -1.0, -1.0,
            -8.0, -8.0, -5.0,
            -20.0, -20.0, -20.0,
            0.0, 0.0, 0.0, 0.0,
            MIN_DURATION,
        ],
        dtype=jnp.float32,
    )
    x_max = jnp.array(
        [
            30.0, 20.0, 8.0,
            1.0, 1.0, 1.0, 1.0,
            8.0, 8.0, 5.0,
            20.0, 20.0, 20.0,
            1.0, 1.0, 1.0, 1.0,
            MAX_DURATION,
        ],
        dtype=jnp.float32,
    )
    constraints = combine_constraints(
        make_state_box_constraints(x_min, x_max),
        make_control_box_constraints(u_min, u_max),
        make_terminal_constraint(goal, half_width=0.835),
    )

    return GenericMPC(
        SLSConfig(
            max_sls_iterations=1,
            sls_primal_tol=1e-2,
            enable_fastsls=True,
            initialize_nominal=True,
            max_initial_sqp_iterations=0,
            warm_start=True,
            rti=True,
            gradient_window=0,
        ),
        SQPConfig(
            max_sqp_iterations=1,
            warm_start=True,
            feas_tol=1e-4,
            step_tol=1e-4,
            line_search=False,
        ),
        ADMMConfig(
            eps_abs=5e-2,
            eps_rel=1e-3,
            max_iterations=250,
            rho_update_frequency=25,
            initial_rho=1.0,
        ),
        config=config,
        dynamics=min_time_dynamics,
        constraints=constraints,
        obstacles=jnp.zeros((0, 3), dtype=jnp.float32),
        cost=min_time_cost,
        disturbance=disturbance,
        shift=1,
        X_in=reference,
        U_in=jnp.full((HORIZON, NU), HOVER_COMMAND, dtype=jnp.float32),
    )



"""Run minimum-time optimization and simulation in receding horizon."""
sim = initialize_hover()
gate_centers = np.asarray([gate.position for gate in GATES], dtype=np.float32)
if not len(gate_centers):
    raise ValueError("At least one gate must be defined in drone_sim.GATES")

# Finish one metre beyond the final gate so the vehicle flies through it.
exit_point = gate_centers[-1] + np.array([1.0, 0.0, 0.0], dtype=np.float32)
waypoints = np.vstack((gate_centers, exit_point))
goal = jnp.asarray(exit_point)
duration = min(MAX_DURATION, max(2.0, float(np.linalg.norm(exit_point)) / 1.5))
state0 = measured_state(duration)
reference0 = build_reference(state0, waypoints, duration)
mpc_controller = build_controller(state0, reference0, goal)
gate_index = 0
simulated_time = 0.0

In [ ]:
controller = copy.deepcopy(mpc_controller)
controller.U0 = jnp.full((HORIZON, NU), HOVER_COMMAND, dtype=jnp.float32)
controller.X0 = reference0

state = state0
render()
import time
time.sleep(5)
for i in range(150):
    x = measured_state(duration)
    remaining = waypoints[gate_index:]
    reference = build_reference(x, remaining, duration)

    solve_start = perf_counter()
    _, predicted_x, predicted_u, _, _, _, _ = controller.run(
        x0=x,
        reference=reference,
        parameter=1.0 / HORIZON,
    )
    solve_ms = 1e3 * (perf_counter() - solve_start)
    optimized_duration = float(np.asarray(predicted_x[0, -1]))
    # A single RTI SQP step can be infeasible during initial warm-up.
    # Bound the duration update so one poor iterate cannot create a very
    # large or tiny physics interval.
    duration = float(
        np.clip(
            optimized_duration,
            max(MIN_DURATION, 0.8 * duration),
            min(MAX_DURATION, 1.2 * duration),
        )
    )
    dt = duration / HORIZON

    command = np.clip(
        np.asarray(predicted_u[0]),
        0.0,
        1.0,
    ).astype(np.float32) * MAX_ROTOR_RPM
    physics_steps = max(1, round(dt * sim.freq))
    state = step(command, n_steps=physics_steps)
    render()
    simulated_time += physics_steps / sim.freq
    position = np.asarray(state.pos[0, 0])

    # if gate_index < len(gate_centers):
    #     if np.linalg.norm(position - gate_centers[gate_index]) <= GATE_TOLERANCE:
    #         print(f"Passed gate {gate_index + 1}")
    #         gate_index += 1

    goal_error = float(np.linalg.norm(position - exit_point))
    print(
        f"RTI {i:03d} | solve {solve_ms:7.2f} ms | "
        f"T {duration:5.2f} s | goal error {goal_error:5.2f} m"
    )
